# Prerequisite
Service Principal used in this notebook must be
- Foundry Developer on Foundry Project Target
- Foundry User on Foundry resource (parent of the project)

In [1]:
%pip install "azure-ai-projects>=2.2.0" azure-identity openai azure-keyvault-secrets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.6/349.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.1/347.1 kB 21.8 MB/s eta 0:00:00
  Attempting uninstall: azure-core
    Found existing installation: azure-core 1.29.4
    Uninstalling azure-core-1.29.4:
      Successfully uninstalled azure-core-1.29.4
Note: you may need to restart the kernel to use updated packages.


In [2]:
dataset_name = "eval_icecreamoperator_minimal"#"EvaluationAgent_testset_wcontext"
evaluation_id = None#'eval_0f07a91c1d8c4217ae32bc98284e7200'

agent_name = "IceCreamOperator"
agent_version = "19"

project_endpoint = "https://ice-foundry-01.services.ai.azure.com/api/projects/foundry-proj-01" # The Azure AI Project project endpoint, as found in the Home page of your Microsoft Foundry portal.
model_deployment_name = "gpt-4.1-1" # The deployment name of the AI model, as found under the "Build" page in the "Models" tab in your Foundry project.

In [3]:
# ## SP for authentication to the Foundry Project
# tenant_id = "94e2fbfd-171c-4f8c-9b70-fbe8002dac58"
# client_id = "e6158f09-b7d6-426f-8613-711fd3d2c6f2"

# connection_id = "dbd10a03-e035-40bb-87ed-3355ebf18687" # connection name: "gelatai-in-keyvault admin"
# client_secret = notebookutils.credentials.getSecretWithConnection(connection_id, "sp-fabriceval-secret")

In [6]:
import time
from azure.core.credentials import AccessToken, TokenCredential
from azure.keyvault.secrets import SecretClient

## SP for authentication to the Foundry Project
tenant_id = "94e2fbfd-171c-4f8c-9b70-fbe8002dac58"
client_id = "e6158f09-b7d6-426f-8613-711fd3d2c6f2"

# --- Key Vault (behind the private endpoint) ---
key_vault_url = "https://gelatai-in-keyvault.vault.azure.net/"  # the vault your connection pointed at
secret_name   = "sp-fabriceval-secret"

# Reuse the Fabric executing identity (workspace identity in pipeline runs / your user interactively)
class FabricTokenCredential(TokenCredential):
    def get_token(self, *scopes, **kwargs):
        token = notebookutils.credentials.getToken("keyvault")   # AAD token for the KV audience
        return AccessToken(token, int(time.time()) + 30 * 60)    # let the SDK refresh well before 1h expiry

secret_client = SecretClient(vault_url=key_vault_url, credential=FabricTokenCredential())
client_secret = secret_client.get_secret(secret_name).value

In [4]:
import os
import json
import time
from datetime import datetime
from pprint import pprint
from pathlib import Path
from typing import Any, List, Dict, Iterable, Union
from packaging.version import Version

from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import DatasetVersion
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)

# Run Evaluation

In [5]:
with ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret) as credential:
    with AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client:
        print("Retrieving existing dataset")

        testset_version = None
        all_existing_versions_iter = project_client.datasets.list_versions(dataset_name)
        all_existing_versions = list(all_existing_versions_iter)

        if not all_existing_versions:
            raise Exception("Testset not found")
        else:
            dataset = max(all_existing_versions, key=lambda d: Version(d.version))
        #dataset = project_client.datasets.get(dataset_name, dataset_version)

        print("Creating an OpenAI client from the AI Project client")
        client = project_client.get_openai_client()

        data_source_config = {
                "type": "custom",
                "item_schema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string"},
                        "response": {"type": "string"},
                        "context": {"type": "string"},
                        "ground_truth": {"type": "string"},
                        # "system_prompt": {"type": "string"}
                    },
                    "required": [],
                },
                "include_sample_schema": True
            }
            
        testing_criteria = [
            {
            "type": "azure_ai_evaluator",
            "id": "ToolCallAccuracy_db7ed8c1-0ef1-4f01-a35e-8758e2a7818f",
            "name": "ToolCallAccuracy",
            "evaluator_name": "builtin.tool_call_accuracy",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}",
                "threshold": 3
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_items}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "QualityGrader_0b88d37e-e7bf-446b-b9fe-63ced5b9f538",
            "name": "QualityGrader",
            "evaluator_name": "builtin.quality_grader",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}"
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "Fluency_d1c4626f-7490-4a3d-9b14-2d75ab73a7cf",
            "name": "Fluency",
            "evaluator_name": "builtin.fluency",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}",
                "threshold": 3
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "Coherence_8dec5bc1-a8c7-488e-b20c-ac7013bdbad1",
            "name": "Coherence",
            "evaluator_name": "builtin.coherence",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}",
                "threshold": 3
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "ToolSelection_2aff19a9-605c-420f-ac53-088d9e9d5747",
            "name": "ToolSelection",
            "evaluator_name": "builtin.tool_selection",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}"
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_items}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "TaskCompletion_80c740da-3d02-4cab-b713-9995c68a01c5",
            "name": "TaskCompletion",
            "evaluator_name": "builtin.task_completion",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}"
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "TaskAdherence_214b4472-76de-476c-a8f1-ee6140ae5658",
            "name": "TaskAdherence",
            "evaluator_name": "builtin.task_adherence",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}",
                "threshold": 3
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_items}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "IntentResolution_442f15c4-9508-4499-a3f9-36e47ff9c586",
            "name": "IntentResolution",
            "evaluator_name": "builtin.intent_resolution",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}",
                "threshold": 3
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "RubricOperator_cf683e0d-cc83-4f00-b506-3ffd9a89687e",
            "name": "RubricOperator",
            "evaluator_name": "RubricOperator",
            "evaluator_version": "",
            "initialization_parameters": {
                "model": f"{model_deployment_name}"
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "id": "Similarity_52155165-ea59-4a6a-8747-e44ad28255af",
            "name": "Similarity",
            "evaluator_name": "builtin.similarity",
            "evaluator_version": "",
            "initialization_parameters": {
                "deployment_name": f"{model_deployment_name}",
                "threshold": 3
            },
            "data_mapping": {
                "query": "{{item.query}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.ground_truth}}",
                "tool_calls": "{{sample.tool_calls}}",
                "tool_definitions": "{{sample.tool_definitions}}"
            }
        }
        ]

        for item in testing_criteria:
            item["data_mapping"]["tool_calls"] = "{{sample.tool_calls}}"
            item["data_mapping"]["tool_definitions"] = "{{sample.tool_definitions}}"

        if not evaluation_id:
            print("Creating Eval Group")
            eval_object = client.evals.create(
                name="EvaluationGroup" + "-" + datetime.utcnow().strftime("%Y%m%d%H%M"),
                data_source_config=data_source_config,
                testing_criteria=testing_criteria,
            )
            print(f"Eval Group created: {eval_object.id}")

            evaluation_id = eval_object.id
        else:
            eval_object = client.evals.retrieve(evaluation_id)

        print("Get Eval Group by Id")
        eval_object_response = client.evals.retrieve(eval_object.id)
        # print("Eval Group Response:")
        # pprint(eval_object_response)

        print("Creating Eval Run with Dataset ID")

        data_source={
                "type": "azure_ai_target_completions",
                "input_messages": {
                    "type": "template",
                    "sample": {
                        "type": "object",
                        "properties": {
                            "output_text": {
                                "type": "string"
                            }
                        }
                    }
                },
                "source": {
                    "type": "file_id",
                    "id": dataset.id
                }
            }

        name = agent_name + "-" + datetime.utcnow().strftime("%Y%m%d%H%M")
        data_source['input_messages']['template'] = [
                            {
                                "role": "user",
                                "content": "{{item.query}}",
                                "type": "message"
                            }
                        ]
        data_source['target'] = {
                        "type": "azure_ai_agent",
                        "name": agent_name,
                        "version": agent_version,
                        "tool_descriptions": []
                    }

        eval_run_object = client.evals.runs.create(
            eval_id=eval_object.id,
            name=name,
            metadata={"team": "eval-exp", "scenario": "dataset-id-v1"},
            data_source=data_source
        )

        print(f"Eval Run created: {eval_run_object.id} at {datetime.utcnow()}")
        # pprint(eval_run_object)

        evaluation_id = eval_object.id
        evaluation_run_id = eval_run_object.id

        print("Get Eval Run by Id")
        eval_run_response = client.evals.runs.retrieve(
            run_id=eval_run_object.id,
            eval_id=eval_object.id,
        )
        # print("Eval Run Response:")
        # pprint(eval_run_response)

        # Poll until the run completes or fails
        while True:
            run = client.evals.runs.retrieve(
                run_id=eval_run_response.id, eval_id=eval_object.id
            )
            if run.status in ("completed", "failed"):
                print(f"Finished at {datetime.utcnow()}")
                output_items = list(
                    client.evals.runs.output_items.list(
                        run_id=run.id, eval_id=eval_object.id
                    )
                )
                # pprint(output_items)
                print(f"Eval Run Report URL: {run.report_url}")
                break

            time.sleep(30)
            print("\033[K", end="\r")
            print(f"Waiting for eval run to complete...{datetime.utcnow()}", end="\r", flush=True)

Retrieving existing dataset
Creating an OpenAI client from the AI Project client
Creating Eval Group
Eval Group created: eval_233a1a5d4df94967bbc6d4ead84ff4db
Get Eval Group by Id
Creating Eval Run with Dataset ID
Eval Run created: evalrun_49de836bdd50467094192edfc4ce7bfa at 2026-06-30 14:14:35.381522
Get Eval Run by Id
Finish at 2026-06-30 14:27:57.643114026-06-30 14:27:56.946724
Eval Run Report URL: https://ai.azure.com/nextgen/r/HgVZNyNGS6ScEaCwWgl3pw,RG-Foundry-01,,ice-foundry-01,foundry-proj-01/build/evaluations/eval_233a1a5d4df94967bbc6d4ead84ff4db/run/evalrun_49de836bdd50467094192edfc4ce7bfa


In [3]:
print(f"Retrieving data for: {project_endpoint}, {evaluation_id}, {evaluation_run_id}")
notebookutils.notebook.run("evalFoundryGetRuns", 300, {"project_endpoint": project_endpoint, "eval_id":evaluation_id, "eval_run_id":evaluation_run_id})

Retrieving data for: https://ice-foundry-01.services.ai.azure.com/api/projects/foundry-proj-01, eval_233a1a5d4df94967bbc6d4ead84ff4db, evalrun_49de836bdd50467094192edfc4ce7bfa


''